In [1]:
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.order import Order
from ibapi.contract import Contract
import threading
import time

class LiquidateClient(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self._connected = threading.Event()
        self._positions = []
        self._positions_done = threading.Event()

    def nextValidId(self, orderId):
        self._next_id = orderId
        self._connected.set()

    def position(self, account, contract, pos, avgCost):
        if pos != 0:
            self._positions.append((account, contract, pos))

    def positionEnd(self):
        self._positions_done.set()

    def error(self, reqId, errorCode, errorString, advancedOrderRejectJson=""):
        if errorCode not in (2104, 2106, 2158, 2119, 2176):
            print(f"Error {errorCode}: {errorString}")

    def orderStatus(self, orderId, status, filled, remaining, avgFillPrice,
                    permId, parentId, lastFillPrice, clientId, whyHeld, mktCapPrice):
        print(f"Order {orderId}: {status} filled={filled} avg={avgFillPrice}")

# Connect
client = LiquidateClient()
client.connect("127.0.0.1", 7497, clientId=99)  # use unique clientId
threading.Thread(target=client.run, daemon=True).start()
client._connected.wait(timeout=10)

# Request all positions
client.reqPositions()
client._positions_done.wait(timeout=10)

# Sell each position
for account, contract, pos in client._positions:
    order = Order()
    order.action = "SELL" if pos > 0 else "BUY"  # reverse the position
    order.totalQuantity = abs(pos)
    order.orderType = "MKT"
    order.eTradeOnly = ""
    order.firmQuoteOnly = ""

    oid = client._next_id
    client._next_id += 1

    print(f"Closing: {order.action} {abs(pos)}x {contract.symbol} "
          f"{contract.secType} {contract.strike} {contract.right}")
    client.placeOrder(oid, contract, order)

time.sleep(10)  # wait for fills
client.disconnect()


Closing: SELL 100.0x NVDA STK 0.0 
Closing: SELL 1.0x SHAK OPT 96.0 C
Closing: SELL 1.0x MMYT OPT 50.0 C
Closing: SELL 100.0x GOOGL STK 0.0 
Closing: SELL 1.0x DXCM OPT 68.0 C
Closing: BUY 100.0x META STK 0.0 
Closing: BUY 100.0x AMZN STK 0.0 
Closing: SELL 100.0x TSLA STK 0.0 
Error 321: Error validating request.-'bG' : cause - Missing order exchange.
Error 321: Error validating request.-'bG' : cause - Missing order exchange.
Error 321: Error validating request.-'bG' : cause - Missing order exchange.
Error 10349: Order TIF was set to DAY based on order preset.
Error 10311: This order will be directly routed to NASDAQ. Direct routed orders may result in higher trade fees. Restriction is specified in Precautionary Settings of Global Configuration/API.
Error 10349: Order TIF was set to DAY based on order preset.
Error 10311: This order will be directly routed to NASDAQ. Direct routed orders may result in higher trade fees. Restriction is specified in Precautionary Settings of Global Conf

In [2]:
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
import threading

class PositionClient(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self._connected = threading.Event()
        self._done = threading.Event()

    def nextValidId(self, orderId):
        self._connected.set()

    def position(self, account, contract, pos, avgCost):
        if pos != 0:
            print(f"{contract.symbol} | {contract.secType} | {contract.right} "
                  f"{contract.strike} | exp={contract.lastTradeDateOrContractMonth} "
                  f"| pos={pos} | avgCost={avgCost:.2f}")

    def positionEnd(self):
        print("\n-- Done --")
        self._done.set()

    def error(self, reqId, errorCode, errorString, advancedOrderRejectJson=""):
        if errorCode not in (2104, 2106, 2158, 2119, 2176):
            print(f"Error {errorCode}: {errorString}")

client = PositionClient()
client.connect("127.0.0.1", 7497, clientId=99)
threading.Thread(target=client.run, daemon=True).start()
client._connected.wait(timeout=10)

client.reqPositions()
client._done.wait(timeout=10)
client.disconnect()


NVDA | STK |  0.0 | exp= | pos=100.0 | avgCost=185.55
SHAK | OPT | C 96.0 | exp=20260313 | pos=1.0 | avgCost=340.05
MMYT | OPT | C 50.0 | exp=20260320 | pos=1.0 | avgCost=185.05
GOOGL | STK |  0.0 | exp= | pos=100.0 | avgCost=307.70
DXCM | OPT | C 68.0 | exp=20260313 | pos=1.0 | avgCost=120.05
META | STK |  0.0 | exp= | pos=-100.0 | avgCost=651.84
AMZN | STK |  0.0 | exp= | pos=-100.0 | avgCost=214.31
TSLA | STK |  0.0 | exp= | pos=100.0 | avgCost=410.31

-- Done --
